# viz_utils — Example Notebook

A single walk-through of every utility in this repo, using synthetic data that mirrors the **actual datasets** from the original project:

| Section | Utility | Data it mirrors |
|---|---|---|
| 1 | `stacked_hbar` | `2019_pol_journalist_retweets.csv` — BJP vs non-BJP RTs per journalist |
| 2 | `labeled_scatter` | `journalists to visualize.xlsx` — followers vs engagement bubble chart |
| 3 | `event_timeline` | `polarity_11March.csv` weekly BJP/INC retweet series |
| 4 | `scatter_line_timeline` | Trending hashtag hourly volume (`viz.ipynb`) |
| 5 | `build_graph` / `draw_network` | `Scripts_viz.ipynb` sourceID→targetID retweet network |
| 6 | `make_wordcloud` | Tweet text corpus (Topic_Modelling.ipynb) |
| 7 | `comparison_cloud` | `celebs_INC.csv` LDA topic × word matrix (comp_cloud.R) |

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from utils.bar import stacked_hbar, vbar
from utils.scatter import labeled_scatter, log_transform
from utils.timeline import event_timeline, scatter_line_timeline, make_month_tick_labels
from utils.network import build_graph, draw_network, degree_to_size
from utils.wordcloud_utils import make_wordcloud, plot_wordcloud, comparison_cloud, lda_topics_to_freqs

rng = np.random.default_rng(42)

---
## 1 · Stacked bar — BJP vs non-BJP retweets per journalist

In [ ]:

journalists = [
    'ShekharGupta','sardesairajdeep','BDUTT','svaradarajan','ravishndtv',
    'RanaAyyub','suhasinih','abhisar_sharma','rahulkanwal','waglenikhil',
    'smitaprakash','AdityaRajKaul','abhijitmajumder','ShivAroor','bainjal',
]
bjp_rts   = rng.integers(5,  400, len(journalists))
other_rts = rng.integers(5,  400, len(journalists))

# Sort by total (mirrors original visual ordering)
order = (bjp_rts + other_rts).argsort()
journalists = [journalists[i] for i in order]
bjp_rts, other_rts = bjp_rts[order], other_rts[order]

fig, ax = stacked_hbar(
    labels=journalists,
    segments=[bjp_rts, other_rts],
    colors=['darkorange', 'skyblue'],
    segment_labels=['BJP RTs', 'Non-BJP RTs'],
    xlabel='Retweets by politicians',
    title='Journalist retweet polarity — BJP vs non-BJP (2019)',
    figsize=(10, 7),
)
plt.tight_layout()
plt.show()

---
## 2 · Bubble scatter — journalist followers vs engagement

In [ ]:
# Mirrors the actual journalist dataframe shape
df_js = pd.DataFrame({
    'Handle':      ['ShekharGupta','sardesairajdeep','BDUTT','svaradarajan','suhasinih',
                    'ravishndtv','RanaAyyub','waglenikhil','rahulkanwal','smitaprakash',
                    'vikramchandra','sagarikaghose','ShivAroor','nistula','abhisar_sharma'],
    'followers':   [2215505,8871256,7068088,544143,1282035,
                    1053095,876194,840941,4428992,751123,
                    2993692,4117016,898206,129627,1050000],
    'pol_following':[1338,1309,1229,1169,999,
                     725,874,679,986,665,
                     748,909,621,789,800],
    'media_type':  ['Print','Television','Television','Digital','Television',
                    'Television','Digital','Print','Television','Print',
                    'Television','Television','Digital','Print','Digital'],
})

color_map = {
    'Digital':    'skyblue',
    'Television': 'violet',
    'Print':      'green',
    'Commentary': 'gold',
}

# Log-transform both axes (mirrors log_f1, log_f2 columns)
x_log = log_transform(df_js['followers'])
y_log = log_transform(df_js['pol_following'])

fig, ax = labeled_scatter(
    x=x_log,
    y=y_log,
    labels=df_js['Handle'],
    categories=df_js['media_type'],
    color_map=color_map,
    xlabel='log₁₀(Followers)',
    ylabel='log₁₀(Politicians followed)',
    title='Top journalists — reach vs. political network',
    left_labels=['svaradarajan','nistula','smitaprakash'],
    right_labels=['sardesairajdeep','BDUTT','rahulkanwal'],
    figsize=(12, 9),
)

# Relabel log ticks with human-readable follower counts (mirrors original)
fig.canvas.draw()
ax.set_xticklabels([
    f'{int(10**float(t.get_text())):,}' if t.get_text() else ''
    for t in ax.get_xticklabels()
])
plt.tight_layout()
plt.show()

---
## 3 · Event timeline — weekly BJP vs INC retweet volume



In [ ]:
weeks = pd.date_range('2019-01-01', periods=104, freq='W')   # 2 years
df_weekly = pd.DataFrame({
    'week':  range(104),
    'BJP':   np.abs(rng.integers(400, 8000, 104) + np.sin(np.arange(104)/8)*1000).astype(int),
    'INC':   np.abs(rng.integers(100, 3000, 104) + np.cos(np.arange(104)/8)*500).astype(int),
    'month': [w.strftime('%b') for w in weeks],
})

# Compress repeated month labels (mirrors make_month_tick_labels usage)
ls = make_month_tick_labels(df_weekly['month'])

# Real events from build_timeline.ipynb
events = {10: 'Budget', 18: 'Elections announced', 28: 'Pulwama',
          40: 'Election results', 55: 'CAA', 72: 'COVID-19', 90: 'Farm Bills'}

fig, axes = event_timeline(
    df=df_weekly,
    x_col='week',
    y_cols=['BJP', 'INC'],
    colors=['darkorange', 'teal'],
    y_labels=['Retweets of BJP politicians', 'Retweets of INC politicians'],
    events=events,
    event_y_offsets=[7000, 2500],
    x_tick_labels=ls,
    title='Weekly Retweet Volume — BJP vs INC (2019–2021)',
    figsize=(14, 6),
)
plt.show()

---
## 4 · Scatter + line — trending hashtag volume over time



In [ ]:
hashtags = [
    '#TwitterStopHoldingAccounts', '#CongressWithRioters', 'Rihanna',
    '#FarmersProtest', '#IndiaAgainstPropaganda', '#IndianFarmersHumanRights', 'Greta',
]
hours = range(72)   # 2nd Feb to 4th Feb, hourly

records = []
for h in hours:
    for tag in hashtags:
        base = 3000 if 'Farmers' in tag else 1000
        spike = 4000 if (tag == 'Rihanna' and 20 < h < 30) else 0
        records.append({
            'group': h, 'name': tag,
            'tweet_volume': int(rng.integers(100, base) + spike),
            'category': 'support' if tag in ['#FarmersProtest','Rihanna','Greta','#IndianFarmersHumanRights']
                        else 'oppose',
        })

df_trending = pd.DataFrame(records)

# Annotate only peak of select hashtags
annotate_tags = ['#FarmersProtest', 'Rihanna', '#IndiaAgainstPropaganda']
peak_mask = (df_trending.groupby('name')['tweet_volume']
               .transform('max') == df_trending['tweet_volume'])
ann_mask = peak_mask & df_trending['name'].isin(annotate_tags)

fig, ax = scatter_line_timeline(
    df=df_trending,
    x_col='group',
    y_col='tweet_volume',
    hue_col='category',
    annotate_mask=ann_mask,
    annotate_col='name',
    xlabel='Hour (2nd Feb → 4th Feb)',
    ylabel='Number of tweets with hashtag',
    title='Trending Hashtag Volume — Farmers Protest coverage',
    figsize=(13, 5),
)
plt.show()

---
## 5 · Retweet network — journalists ↔ politicians


In [ ]:
# Mirrors the sourceID→target edge structure from Scripts_viz.ipynb
pol_nodes  = ['narendramodi','AmitShah','RahulGandhi','ArvindKejriwal',
               'ShashiTharoor','smritiirani','OmarAbdullah','asadowaisi']
jour_nodes = ['ShekharGupta','BDUTT','ravishndtv','abhisar_sharma',
               'rahulkanwal','svaradarajan','RanaAyyub','AdityaRajKaul']

edges_list = []
for _ in range(90):
    edges_list.append({
        'source': rng.choice(jour_nodes),
        'target': rng.choice(pol_nodes),
        'weight': int(rng.integers(1, 30)),
    })
df_edges = pd.DataFrame(edges_list)

G = build_graph(df_edges, source_col='source', target_col='target', weight_col='weight')

# Colour: journalists = steelblue, BJP pols = darkorange, others = teal
bjp = {'narendramodi', 'AmitShah', 'smritiirani'}
color_map_net = {}
for n in G.nodes():
    if n in jour_nodes:  color_map_net[n] = 'steelblue'
    elif n in bjp:       color_map_net[n] = 'darkorange'
    else:                color_map_net[n] = 'teal'

legend_patches = [
    mpatches.Patch(color='steelblue',  label='Journalist'),
    mpatches.Patch(color='darkorange', label='BJP politician'),
    mpatches.Patch(color='teal',       label='Other politician'),
]

fig, ax = draw_network(
    G,
    node_color_map=color_map_net,
    node_size=degree_to_size(G, scale=20, min_size=40),
    layout='spring',
    layout_kwargs={'seed': 7, 'k': 0.8},
    title='Journalist → Politician Retweet Network',
    legend_patches=legend_patches,
    figsize=(10, 10),
)
plt.show()

---
## 6 · Word cloud — tweet corpus



In [ ]:
# Mirrors tweet text corpus from the DB query in Topic_Modelling.ipynb
corpus_text = """
india farmers protest democracy rights congress modi bjp government
rahul gandhi india today parliament agriculture farmers economy
modi ji india bjp growth development infrastructure education
farmers protest rights solidarity india democracy parliament law
congress india rahul vote election 2019 campaign rally
india bjp modi development growth economy jobs
farmers agriculture MSP procurement support government policy
parliament session budget india congress bjp vote
india today tomorrow future development youth jobs education
protest solidarity rights democracy india farmers support
"""

wc = make_wordcloud(
    text=corpus_text,
    extra_stopwords={'india', 'amp', 'ji', 'will'},   # same noise words removed in original
    max_words=80,
    colormap='RdYlGn',
)
fig, ax = plot_wordcloud(wc, title='Political tweet corpus — word cloud', figsize=(12, 5))
plt.show()

---
## 7 · Comparison cloud — LDA topics (port of comp_cloud.R)



In [ ]:
# Mirrors actual topic words from Topic_Modelling.ipynb gensim output
# (word probabilities from model.top_topics)
topic_freqs = {
    'Topic 1': {  # rally / campaign theme
        'india': 0.041, 'by': 0.041, 'are': 0.038, 'from': 0.036,
        'today': 0.023, 'congress': 0.030, 'my': 0.013, 'have': 0.012,
        'farmers': 0.025, 'rally': 0.019, 'vote': 0.015,
    },
    'Topic 2': {  # personal / engagement theme
        'of': 0.192, 'with': 0.170, 'my': 0.145, 'here': 0.127,
        'on': 0.088, 'and': 0.068, 'we': 0.041, 'amp': 0.035,
        'modi': 0.009, 'ji': 0.008, 'our': 0.005, 'meet': 0.020,
    },
    'Topic 3': {  # critique / opposition
        'is': 0.145, 'and': 0.123, 'it': 0.095, 'you': 0.071,
        'ji': 0.060, 'modi': 0.050, 'this': 0.048, 'our': 0.048,
        'congress': 0.030, 'corruption': 0.022, 'question': 0.018,
    },
    'Topic 4': {  # farmers / protest
        'you': 0.250, 'for': 0.138, 'are': 0.105, 'all': 0.091,
        'amp': 0.050, 'india': 0.042, 'our': 0.036, 'modi': 0.031,
        'farmers': 0.035, 'protest': 0.028, 'rights': 0.022,
    },
    'Topic 5': {  # development / economy
        'of': 0.172, 'amp': 0.130, 'in': 0.119, 'will': 0.091,
        'this': 0.055, 'congress': 0.027, 'bjp': 0.023,
        'economy': 0.031, 'growth': 0.025, 'infrastructure': 0.019,
    },
    'Topic 6': {  # Hindi-language tweets
        'be': 0.114, 'will': 0.081, 'amp': 0.074, 'in': 0.056,
        'and': 0.055, 'congress': 0.049, 'this': 0.049, 'that': 0.048,
        'parliament': 0.030, 'session': 0.027, 'democracy': 0.021,
    },
}

# Mirrors brewer.pal(6, "Paired") from comp_cloud.R
paired_colors = ['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c', '#fb9a99', '#e31a1c']

fig, axes = comparison_cloud(
    topic_freqs=topic_freqs,
    colors=paired_colors,
    max_words=100,
    title_size=13,
    figsize=(15, 9),
    ncols=3,
)
fig.suptitle('Comparison Cloud — LDA Topics (celebs_INC corpus)', fontsize=14, y=1.01)
plt.show()

---
### Using with real LDA output

```python
# If you have a trained gensim model:
from utils.wordcloud_utils import lda_topics_to_freqs, comparison_cloud

topic_freqs = lda_topics_to_freqs(model.top_topics(corpus))
comparison_cloud(topic_freqs, max_words=100)
```

### Using with the CSV from comp_cloud.R

```python
import pandas as pd
df = pd.read_csv('celebs_INC.csv', index_col='word')
# df columns = ['Topic 1', 'Topic 2', ...], rows = words
topic_freqs = {col: df[col].to_dict() for col in df.columns}
comparison_cloud(topic_freqs, max_words=100)
```